In [1]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from typing import Optional, List
import time
import os

In [2]:
class PersonDetails(BaseModel):
    first_name: str = Field(default="Unknown", description="First name.")
    last_name: str = Field(default="Unknown", description="Last name.")
    address: str = Field(default="Unknown", description="Full address.")
    phone: str = Field(default="Unknown", description="Phone number.")

class SuspectDescription(BaseModel):
    race: str = Field(default="Unknown", description="Race.")
    gender: str = Field(default="Unknown", description="Gender.")
    clothing: str = Field(default="Unknown", description="Clothing.")
    physical_features: str = Field(default="Unknown", description="Features.")

class VehicleDescription(BaseModel):
    make: str = Field(default="Unknown", description="Make.")
    model: str = Field(default="Unknown", description="Model.")
    color: str = Field(default="Unknown", description="Color.")
    plate_number: str = Field(default="Unknown", description="Plate.")

In [3]:
class IncidentReport(BaseModel):
    reporting_person: PersonDetails = Field(default_factory=PersonDetails)
    incident_type: str = Field(default="Unknown", description="Crime type.")
    when_time: str = Field(default="Unknown", description="Date/Time.")
    where_location: str = Field(default="Unknown", description="Location.")
    what_happened: str = Field(default="Unknown", description="Narrative.")
    why_motive: str = Field(default="Unknown", description="Motive.")
    how_method: str = Field(default="Unknown", description="Method.")
    witness_info: str = Field(default="Unknown", description="Witnesses.")
    suspect_info: SuspectDescription = Field(default_factory=SuspectDescription)
    vehicle_info: VehicleDescription = Field(default_factory=VehicleDescription)

In [4]:
@tool(args_schema=IncidentReport)
def submit_incident_report(**kwargs):
    """
    PERMANENTLY CLOSES THE CASE.
    ONLY call this function AFTER the user has explicitly confirmed the summary.
    This generates the final .txt file.
    """
   

    timestamp = int(time.time())
    filename = f"Incident_Report_{timestamp}.txt"
    filepath = os.path.join(".", filename)
    
    try:
        report_data = IncidentReport(**kwargs)
        with open(filepath, "w") as f:
            f.write(str(report_data.model_dump()))
        return f"Case closed. File generated: {filename}"
    except Exception as e:
        return f"ERROR: {e}"
 
   

In [ ]:
system_prompt = """You are a Public Safety Official. Your job and duty is to interview the user to fill out an Incident Report.


*** CONVERSATIONAL RULES ***
1.  **NO JSON**: Never output JSON, code blocks, or dictionary strings to the user. Speak only in natural English.
2.  **ONE STEP AT A TIME**: Do not ask for everything at once.
3.  **CONFIRM & ASK**: After the user provides info, you MUST:
    -   First, confirm what you heard (e.g., "Thank you, Mohamed from 123 Main St."). is the information correct ?
        -   If user says 'correct' move to the next questions. If user says "incorrect" ask the user to correct the information.
   
*** INTERVIEW SCRIPT ***
1.  **Start**: Ask for the user's Full Name and current Address.
2.  **Step 2**: Confirm name/address. Then ask "When and Where did this incident happen?"
3.  **Step 3**: Confirm time/location. Then ask "Please describe exactly what happened."
4.  **Step 4**: Confirm the narrative. Then ask about Suspects or Vehicles.
5.  **Review**: Summarize all collected info and ask "Is this correct?"
6.  **End**: Only when they say "Yes", call the 'submit_incident_report' tool.

2.  **VERIFICATION PHASE**:
    -   Once you have all details, print a summary and ask: "Is this correct?"

3.  **SUBMISSION PHASE**:
    -   ONLY when the user says "Yes" or "Correct", you may call the tool.
    -   **CRITICAL JSON RULE**: When calling the tool, 'reporting_person' MUST be a dictionary object, NOT a string.
        -   CORRECT: reporting_person={'first_name': 'Joe', 'last_name': 'Doe', ...}
        -   WRONG: reporting_person='Joe Doe'

*** NEGATIVE CONSTRAINTS ***
-   DO NOT call the tool in your first response.
-   DO NOT make up a name like "John Doe".
-   DO NOT submit the report until the user confirms the summary.
"""

In [18]:
model = ChatOllama(model="llama3.1:8b", temperature=0)

checkPointer = InMemorySaver()

agent = create_agent(
    model,
    tools=[submit_incident_report],
    system_prompt=system_prompt,
    checkpointer=checkPointer
)

In [19]:
config = {"configurable": {"thread_id": "case_file_v7"}}

In [20]:
while True:
    user_input = input("\nReportee: ")
    if user_input.lower() in ["exit", "done", "quit"]:
        break

    for step in agent.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
        stream_mode="values"
    ):
        last_message = step["messages"][-1]

        if last_message.type == "ai" and last_message.content:
            print(f"Officer: {last_message.content}")

    
        if last_message.type == "tool":
            print(f"\n[System]: {last_message.content}")


[System]: Case closed. File generated: Incident_Report_1767435040.txt
Officer: Let's start again.

To fill out the incident report, I need some information from you. Can you please tell me your Full Name and current Address?

[System]: Case closed. File generated: Incident_Report_1767435053.txt
Officer: Let's start again.

To fill out the incident report, I need some information from you. You mentioned your name is Mohamed Sharif and your current address is...?

[System]: Case closed. File generated: Incident_Report_1767435077.txt
Officer: Now that I have your name and address, can you please tell me when and where this incident happened?

[System]: Case closed. File generated: Incident_Report_1767435090.txt
Officer: Now that I have your name and address, and the location of the incident, can you please describe exactly what happened? For example, did you see anyone or anything suspicious around your bike?

[System]: Case closed. File generated: Incident_Report_1767435127.txt
Officer: